# 🧠 Chimera Colab Brain v7.1 — Auto-VRAM DEEPSEEK R1 DISTILL (70B/32B) + Qwen3 fallback

One click → this Colab VM becomes the Strix brain: an OpenAI-compatible GPU LLM endpoint, reverse-tunneled to the VPS at `127.0.0.1:8898`, registered with colony `:8766` as a `brain` GPU worker.

**Press Ctrl+F9 (Run All). First run downloads 20-46 GB of GGUF, then loads. The notebook self-reports READY FOR STRIX.**

## v7 auto-VRAM tiers (nvidia-smi → model)
| GPU VRAM | Backend | Model (GGUF) |
|----------|---------|--------------|
| ≥ 70 GB (A100 80GB) | llama.cpp subprocess | **DeepSeek-R1-Distill-Llama-70B Q5_K_M** (46 GB) |
| ≥ 40 GB (A100 40GB) | llama.cpp subprocess | **DeepSeek-R1-Distill-Qwen-32B Q4_K_M** (19.9 GB) |
| < 40 GB (T4/L4) | transformers | Qwen3-30B-YOYO-Opus (bf16 / 4-bit NF4) |

**NOTE:** DeepSeek-R1-Distill-**Qwen**-70B was REMOVED from HF (unsloth/bartowski/mlabonne all gone). The 70B tier therefore uses R1-Distill-**Llama**-70B — same R1 reasoning lineage, Llama-70B base, public repo `unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF`.

## v6.1 → v7.1
- Auto-VRAM model selection at boot
- GGUF tiers served by a llama.cpp subprocess (`python -m llama_cpp.server` on :8002, all layers GPU, n_ctx 8192)
- /v1 API contract **BYTE-IDENTICAL**: bearer gate (BRAIN_API_KEY), global cap, flat usage log, /healthz, /v1/models, streaming SSE, tool_calls
- transformers fallback path unchanged


In [ ]:
# @title ⚙️ CELL 1: Config, deps, GPU detect → auto-VRAM tier (idempotent)
import subprocess, os, time, json, urllib.request, socket, threading, sys

VPS = "187.124.226.128"
TUNNEL_PORT = 8898                          # VPS-side: 127.0.0.1:8898 -> Colab :8001
LLM_PORT = 8001                             # Colab-side: FastAPI listens here
LLAMA_PORT = 8002                           # llama.cpp subprocess server (GGUF tiers)
COLONY_URL = "http://187.124.226.128:8766"
_HN = socket.gethostname()
MACHINE_ID = "colab-brain-" + (_HN.split("-")[0] if "-" in _HN else _HN[:8])

print("=" * 55); print("  CHIMERA COLAB BRAIN v7.2.2 — BOOT"); print("=" * 55)

def _secret(name):
    """One secret path for everything: Colab userdata -> env -> '' (never hardcoded)."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

TOKEN = _secret("COLONY_TOKEN")
AGENT_PASS = _secret("AGENT_PASS")
BRAIN_API_KEY = _secret("BRAIN_API_KEY")

print("[AUTH] COLONY_TOKEN: " + ("OK (" + str(len(TOKEN)) + " chars)" if TOKEN else "MISSING (register will 401 — set it: key icon -> COLONY_TOKEN)"))
print("[AUTH] AGENT_PASS: " + ("OK (" + str(len(AGENT_PASS)) + " chars)" if AGENT_PASS else "MISSING — reverse tunnel to VPS will fail. Set it: key icon -> AGENT_PASS"), flush=True)
print("[AUTH] BRAIN_API_KEY: " + ("OK (" + str(len(BRAIN_API_KEY)) + " chars) — /v1/* requires Bearer" if BRAIN_API_KEY else "MISSING — /v1/* will 401 (fail closed). Set it: key icon -> BRAIN_API_KEY"), flush=True)

try:
    gpu_name = subprocess.check_output("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True).decode().strip().split("\n")[0]
    gpu_vram = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).decode().strip().split("\n")[0])
except Exception:
    gpu_name, gpu_vram = "CPU", 0
print(f"[GPU] {gpu_name} ({gpu_vram} MB)")

# ── v7 auto-VRAM tier ──
VRAM_TIER = "70b" if gpu_vram >= 70000 else ("32b" if gpu_vram >= 40000 else ("14b" if gpu_vram >= 14000 else "qwen3"))
if VRAM_TIER == "70b":
    BACKEND = "llamacpp"
    MODEL_REPO, GGUF_FILE = "unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF", "DeepSeek-R1-Distill-Llama-70B-Q5_K_M.gguf"
    MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-70B"
    MODEL_ALIAS = "deepseek-r1-llama-70b"
    GGUF_GB = 46
elif VRAM_TIER == "32b":
    BACKEND = "llamacpp"
    MODEL_REPO, GGUF_FILE = "unsloth/DeepSeek-R1-Distill-Qwen-32B-GGUF", "DeepSeek-R1-Distill-Qwen-32B-Q4_K_M.gguf"
    MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
    MODEL_ALIAS = "deepseek-r1-qwen-32b"
    GGUF_GB = 20
elif VRAM_TIER == "14b":
    BACKEND = "llamacpp"
    MODEL_REPO, GGUF_FILE = "bartowski/huihui-ai_Qwen3-14B-abliterated-GGUF", "huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf"
    MODEL_ID = "huihui-ai/Qwen3-14B-abliterated"
    MODEL_ALIAS = "qwen3-14b"
    GGUF_GB = 9.2
else:
    BACKEND = "transformers"
    MODEL_ID = "DavidAU/Qwen3-30B-A3B-YOYO-V2-Claude-4.6-Opus-High-INSTRUCT"
    MODEL_ALIAS = "qwen3-30b-yoyo-opus"
print(f"[TIER] {VRAM_TIER} ({gpu_vram}MB) -> backend={BACKEND} alias={MODEL_ALIAS}", flush=True)

print("[1/4] Installing deps (idempotent, no torch reinstall)...")
subprocess.run("apt-get update -qq && apt-get install -y -qq sshpass net-tools >/dev/null 2>&1", shell=True)
if BACKEND == "llamacpp":
    if subprocess.run([sys.executable, "-m", "pip", "show", "llama-cpp-python"], capture_output=True).returncode != 0:
        print("   installing llama-cpp-python (CUDA cu124 wheel)...", flush=True)
        _inst = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"],
                       capture_output=True, text=True)
        if _inst.returncode != 0:
            print("   CUDA wheel install FAILED: " + _inst.stderr[-600:], flush=True)
            print("   Python is " + sys.version.split()[0] + " — no cp313 wheels exist. Source-building with CUDA (~5-10 min, one-time)...", flush=True)
            _env = dict(os.environ)
            _env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
            _inst = subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
                                    "llama-cpp-python"], env=_env, capture_output=True, text=True, timeout=1500)
            if _inst.returncode != 0:
                print("   CUDA source build FAILED: " + _inst.stderr[-800:], flush=True)
            else:
                print("   CUDA source build OK", flush=True)
        try:
            import llama_cpp
            print("   llama_cpp import OK", flush=True)
        except Exception as _ie:
            print("   llama_cpp import FAILED: " + repr(_ie), flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "transformers", "accelerate",
                "safetensors", "sentencepiece", "bitsandbytes", "peft", "huggingface_hub"], capture_output=True)
print("   deps done")
print(f"[ID] {MACHINE_ID} | VPS {VPS} | :{LLM_PORT} -> VPS :{TUNNEL_PORT}")


In [ ]:
# @title 🧠 CELL 2: Load the brain — llama.cpp subprocess (GGUF tiers) / transformers (fallback)
import torch, time

if BACKEND == "llamacpp":
    # ── GGUF download + llama.cpp server subprocess ──
    import subprocess as _sp
    from huggingface_hub import hf_hub_download
    if "LLM_PROC" in globals() and LLM_PROC is not None and LLM_PROC.poll() is None:
        print(f"llama.cpp server already running (pid {LLM_PROC.pid}) — skipping (Run All idempotent)")
    else:
        t0 = time.time()
        print(f"Downloading {GGUF_FILE} (~{GGUF_GB} GB, one-time)...", flush=True)
        GGUF_PATH = hf_hub_download(repo_id=MODEL_REPO, filename=GGUF_FILE)
        print(f"   GGUF at {GGUF_PATH} in {time.time()-t0:.0f}s", flush=True)
        LLM_LOG = open("/content/llama_server.log", "ab")
        def _llama_health():
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{LLAMA_PORT}/health", timeout=2)
                return True
            except Exception:
                return False
        def _start_llama():
            print("Starting llama.cpp server (n_gpu_layers=999, n_ctx=8192)...", flush=True)
            return _sp.Popen([sys.executable, "-m", "llama_cpp.server",
                "--model", GGUF_PATH, "--n_gpu_layers", "999", "--n_ctx", "8192",
                "--host", "127.0.0.1", "--port", str(LLAMA_PORT), "--log_level", "warning"],
                stdout=LLM_LOG, stderr=LLM_LOG, start_new_session=True)
        LLM_PROC = _start_llama()
        ok = False
        for _ in range(120):
            time.sleep(2)
            if _llama_health():
                ok = True
                break
        if ok:
            print(f"llama.cpp serving on :{LLAMA_PORT} in {time.time()-t0:.0f}s (pid {LLM_PROC.pid})", flush=True)
        else:
            print("   llama.cpp did NOT come healthy in 240s — log tail:", flush=True)
            try:
                print(open("/content/llama_server.log","rb").read()[-2500:].decode(errors="replace"), flush=True)
            except Exception as _le:
                print("   (no log file)", flush=True)
            if LLM_PROC.poll() is not None:
                print(f"   subprocess EXITED rc={LLM_PROC.poll()}", flush=True)
        def _llama_watchdog():
            fails = 0
            global LLM_PROC
            while True:
                time.sleep(20)
                if _llama_health():
                    fails = 0
                    continue
                fails += 1
                if fails >= 3:
                    print("[llama watchdog] server unresponsive — restarting subprocess", flush=True)
                    try: LLM_PROC.kill()
                    except Exception: pass
                    time.sleep(2)
                    LLM_PROC = _start_llama()
                    fails = 0
        if not globals().get("_LLAMA_WATCH_STARTED"):
            _LLAMA_WATCH_STARTED = True
            threading.Thread(target=_llama_watchdog, daemon=True).start()
            print("   llama watchdog armed (20s)", flush=True)
else:
    # ── transformers fallback (existing v6.1 path, unchanged) ──
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if "model" in globals() and model is not None:
        print(f"Model already loaded ({getattr(model, 'dtype', '?')}) — skipping reload (Run All idempotent)")
    else:
        print("Loading tokenizer...", flush=True)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        t0 = time.time()
        USE_4BIT = gpu_vram < 70000
        print(f"VRAM {gpu_vram}MB -> {'4-bit NF4 quantized' if USE_4BIT else 'bf16 full precision'}", flush=True)
        if USE_4BIT:
            from transformers import BitsAndBytesConfig
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                quantization_config=bnb, device_map="auto", low_cpu_mem_usage=True)
        else:
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                device_map="auto", low_cpu_mem_usage=True)
        model.eval()
        n = sum(p.numel() for p in model.parameters()) / 1e9
        print(f"Model loaded in {time.time()-t0:.0f}s | {n:.1f}B | dtype={model.dtype} | device={model.device}", flush=True)


In [ ]:
# @title 🖥️ CELL 3: OpenAI-compatible server v6.1 (event loop can NEVER wedge)
# Distilled: no TextIteratorStreamer. Full generation in a worker thread via
# asyncio.to_thread; the loop only awaits. healthz reads a liveness flag.
import json, re, threading, uuid, asyncio, time, urllib.request, torch, os, hmac
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field
import uvicorn

LAST_BEAT = {"t": time.time()}   # set by generation + heartbeat thread; healthz reads ONLY this

app = FastAPI()

@app.get("/")
def root():
    return {"service": "chimera-colab-brain", "version": "v7.2.1", "model": MODEL_ALIAS, "healthz": "/healthz"}

# ── Sovereign bearer-key gate (single operator — NO per-user anything) ──
# healthz stays OPEN so tunnel/watchdog liveness probes keep working.
# /v1/* requires `Authorization: Bearer $BRAIN_API_KEY` (Colab secret).
BRAIN_KEY = globals().get("BRAIN_API_KEY", "")
MAX_REQ = int(os.environ.get("BRAIN_MAX_REQUESTS", "2000"))
REQ_COUNT = {"n": 0}
USAGE_LOG = "/content/brain_usage.log"

def _log_usage(client_ip, p_tok, c_tok):
    try:
        with open(USAGE_LOG, "a") as _f:
            _f.write("%s %s %s prompt=%d completion=%d\n" % (time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), client_ip or "?", MODEL_ALIAS, p_tok, c_tok))
    except Exception:
        pass

@app.middleware("http")
async def _sovereign_gate(request, call_next):
    path = request.url.path
    if path == "/healthz":
        return await call_next(request)
    auth = request.headers.get("authorization", "")
    key = auth[7:].strip() if auth.lower().startswith("bearer ") else ""
    if not BRAIN_KEY or not hmac.compare_digest(key, BRAIN_KEY):
        return JSONResponse({"error": {"message": "unauthorized — BRAIN_API_KEY missing/mismatched (set in Colab secrets)", "type": "auth_error"}}, status_code=401)
    if path == "/v1/chat/completions":
        REQ_COUNT["n"] += 1
        if REQ_COUNT["n"] > MAX_REQ:
            return JSONResponse({"error": {"message": "session request cap %d exceeded" % MAX_REQ, "type": "rate_limit_error"}}, status_code=429)
    resp = await call_next(request)
    resp.headers["X-Brain-Requests"] = str(REQ_COUNT["n"])
    return resp

_THINK_OPEN = re.compile(r"<\|?thinking\|?>")
_THINK_CLOSE = re.compile(r"<\|?/thinking\|?>")
_TAG_JUNK = re.compile(r"<\|?(?:/?)im_(?:start|end)\|?>|<\|(?:endoftext|eot_id)\|?>|<\|?/?thinking\|?>")

def _clean(t): return _TAG_JUNK.sub("", t)

# Qwen3 native tool format is XML: <tool_call><tool_name>..</tool_name><parameters>{json}</parameters></tool_call>
# Some variants emit raw JSON inside the tags — handle both.
_TOOL_XML = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)
def _parse_tool_calls(text):
    calls = []
    for m in _TOOL_XML.finditer(text):
        block = m.group(1).strip()
        obj = None
        nm_x = re.search(r"<tool_name>\s*(.*?)\s*</tool_name>", block, re.S)
        pm_x = re.search(r"<parameters>\s*(.*?)\s*</parameters>", block, re.S)
        if nm_x and pm_x:
            name = nm_x.group(1).strip()
            try: args = json.loads(pm_x.group(1).strip())
            except Exception: args = {"raw": pm_x.group(1).strip()}
            obj = {"name": name, "arguments": args}
        else:
            try:
                j = json.loads(block)
                if isinstance(j, dict) and "name" in j:
                    obj = {"name": j["name"], "arguments": j.get("arguments", {})}
            except Exception:
                obj = None
        if not obj: continue
        name, args = obj["name"], obj["arguments"]
        if isinstance(args, dict): args = json.dumps(args)
        calls.append({"id": "call_" + uuid.uuid4().hex[:24], "type": "function",
                      "function": {"name": name, "arguments": args}})
    return calls

def _apply_chat(messages, enable_thinking, tools=None):
    kw = {"chat_template_kwargs": {"enable_thinking": enable_thinking}}
    if tools: kw["tools"] = tools
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)

def _split(text):
    # (reasoning, content) from full decode. If thinking opens but never closes,
    # tags are stripped by _clean and the answer is never lost.
    reasoning = None
    m = _THINK_OPEN.search(text)
    if m:
        c = _THINK_CLOSE.search(text, m.end())
        if c:
            reasoning = text[m.end():c.start()]
            text = text[c.end():]
    return reasoning, _clean(text)

def _gen_llama(messages, max_new_tokens, temperature, top_p, tools=None):
    # GGUF tiers: call the local llama.cpp OpenAI-compat server (non-stream inner; outer SSE unchanged).
    body = {"model": MODEL_ALIAS, "messages": messages, "max_tokens": int(max_new_tokens),
            "temperature": float(temperature), "top_p": float(top_p), "stream": False}
    if tools:
        body["tools"] = tools
    req = urllib.request.Request("http://127.0.0.1:%d/v1/chat/completions" % LLAMA_PORT,
                                 data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    resp = json.loads(urllib.request.urlopen(req, timeout=900).read())
    msg = (resp.get("choices") or [{}])[0].get("message", {})
    content = msg.get("content") or ""
    calls = []
    for tc in msg.get("tool_calls") or []:
        fn = tc.get("function", {})
        calls.append({"id": tc.get("id", "call_" + uuid.uuid4().hex[:24]), "type": "function",
                      "function": {"name": fn.get("name", ""), "arguments": fn.get("arguments", "{}")}})
    if not calls and tools:
        calls = _parse_tool_calls(content)
    u = resp.get("usage") or {}
    return None, content, calls, u.get("prompt_tokens", 0), u.get("completion_tokens", 0)

def _generate(messages, max_new_tokens, temperature, top_p, enable_thinking, tools=None):
    # FULL generation, runs in a worker thread. Returns (reasoning, content, tool_calls, p_tok, c_tok).
    LAST_BEAT["t"] = time.time()
    if BACKEND == "llamacpp":
        return _gen_llama(messages, max_new_tokens, temperature, top_p, tools)
    text = _apply_chat(messages, enable_thinking, tools)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs,
                             max_new_tokens=int(max_new_tokens),
                             temperature=float(temperature), top_p=float(top_p),
                             do_sample=temperature > 0, repetition_penalty=1.05,
                             pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=False)
    LAST_BEAT["t"] = time.time()
    reasoning, content = _split(decoded)
    calls = _parse_tool_calls(decoded) if tools else []
    return reasoning, content, calls, int(inputs["input_ids"].shape[1]), int(new_tokens.shape[0])

class ChatBody(BaseModel):
    model: str | None = None
    messages: list = Field(default_factory=list)
    temperature: float = 0.7
    top_p: float = 0.9
    max_tokens: int = 1024
    stream: bool = False
    tools: list = None

def _ok_model(m):
    return m in (None, MODEL_ALIAS, MODEL_ID)

@app.get("/healthz")
def healthz():
    return {"status": "ok", "model": MODEL_ALIAS, "last_beat": round(time.time() - LAST_BEAT["t"], 1)}

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [
        {"id": MODEL_ALIAS, "object": "model", "owned_by": "chimera"},
        {"id": MODEL_ID, "object": "model", "owned_by": "chimera"}]}

@app.get("/v1/models/{model_id:path}")
def model_detail(model_id: str):
    if model_id not in (MODEL_ALIAS, MODEL_ID):
        return JSONResponse({"error": {"message": "unknown model", "type": "invalid_request_error"}}, status_code=404)
    return {"id": model_id, "object": "model", "owned_by": "chimera"}

@app.post("/v1/chat/completions")
async def chat(req: ChatBody, request: Request):
    if not _ok_model(req.model):
        return JSONResponse({"error": {"message": "unknown model", "type": "invalid_request_error"}}, status_code=404)
    messages = req.messages or []
    tools = req.tools or None
    enable_thinking = not tools   # thinking off for tool calls = cleaner XML
    if req.stream:
        async def gen():
            cid = "chatcmpl-" + uuid.uuid4().hex[:24]
            yield _sse(cid, {"role": "assistant"})
            reasoning, content, calls, p_tok, c_tok = await asyncio.to_thread(
                _generate, messages, req.max_tokens, req.temperature, req.top_p, enable_thinking, tools)
            _log_usage(request.client.host if request.client else None, p_tok, c_tok)
            if reasoning:
                for i in range(0, len(reasoning), 24):
                    yield _sse(cid, {"reasoning_content": reasoning[i:i+24]})
            if calls:
                for i, c in enumerate(calls):
                    yield _sse(cid, {"tool_calls": [{"index": i, "id": c["id"], "type": "function",
                        "function": {"name": c["function"]["name"], "arguments": c["function"]["arguments"]}}]})
            if content:
                for i in range(0, len(content), 24):
                    yield _sse(cid, {"content": content[i:i+24]})
            yield _sse(cid, {}, finish="tool_calls" if calls else "stop")
            yield "data: [DONE]\n\n"
        return StreamingResponse(gen(), media_type="text/event-stream")
    reasoning, content, calls, p_tok, c_tok = await asyncio.to_thread(
        _generate, messages, req.max_tokens, req.temperature, req.top_p, enable_thinking, tools)
    _log_usage(request.client.host if request.client else None, p_tok, c_tok)
    msg = {"role": "assistant", "content": content or None}
    if reasoning: msg["reasoning_content"] = reasoning
    if calls: msg["tool_calls"] = calls
    return JSONResponse({
        "id": "chatcmpl-" + uuid.uuid4().hex[:24], "object": "chat.completion",
        "created": int(time.time()), "model": MODEL_ALIAS,
        "choices": [{"index": 0, "message": msg,
                     "finish_reason": "tool_calls" if calls else ("stop" if content else "length")}],
        "usage": {"prompt_tokens": p_tok, "completion_tokens": c_tok, "total_tokens": p_tok + c_tok}})

def _sse(cid, delta, finish=None):
    d = {"id": cid, "object": "chat.completion.chunk", "created": int(time.time()), "model": MODEL_ALIAS,
         "choices": [{"index": 0, "delta": delta, "finish_reason": finish}]}
    return "data: " + json.dumps(d) + "\n\n"

def _beat_loop():
    while True:
        LAST_BEAT["t"] = time.time()
        time.sleep(30)
threading.Thread(target=_beat_loop, daemon=True).start()

_UV = {"srv": None, "loop": None}

def _start_server():
    cfg = uvicorn.Config(app, host="0.0.0.0", port=LLM_PORT, log_level="warning")
    srv = uvicorn.Server(cfg)
    loop = asyncio.new_event_loop()
    _UV["srv"], _UV["loop"] = srv, loop
    def run():
        asyncio.set_event_loop(loop)
        loop.run_until_complete(srv.serve())
    threading.Thread(target=run, daemon=True).start()

def _hz_alive():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{LLM_PORT}/healthz", timeout=3)
        return True
    except Exception:
        return False

def _watchdog():
    fails = 0
    while True:
        time.sleep(20)
        if _hz_alive():
            fails = 0
            continue
        fails += 1
        if fails >= 3:
            print("[watchdog] server unresponsive 3x — restarting uvicorn thread", flush=True)
            srv, loop = _UV["srv"], _UV["loop"]
            if srv and loop:
                try: loop.call_soon_threadsafe(srv.should_exit.set)
                except Exception: pass
            time.sleep(4)
            for _ in range(6):
                if not _hz_alive():
                    break
                time.sleep(1)
            try: _start_server()
            except Exception as e: print("[watchdog] restart failed:", e, flush=True)
            fails = 0

if not globals().get("_SRV_STARTED"):
    _SRV_STARTED = True
    threading.Thread(target=_watchdog, daemon=True).start()
    print("   watchdog armed (20s)", flush=True)

if not _hz_alive():
    print("starting server...", flush=True)
    _start_server()
    for _ in range(15):
        time.sleep(1)
        if _hz_alive(): break
print("Brain serving: /healthz -> " + ("200 on :%d" % LLM_PORT if _hz_alive() else "FAIL"), flush=True)

In [ ]:
# @title 🔗 CELL 4: Reverse tunnel (self-healing) + colony registration
import subprocess, time, json, urllib.request, threading, socket, os

print(f"[2/4] Reverse tunnel: VPS 127.0.0.1:{TUNNEL_PORT} -> Colab :{LLM_PORT}")

os.environ["SSHPASS"] = AGENT_PASS
TUNNEL_CMD = ("sshpass -e ssh -o StrictHostKeyChecking=no "
              "-o ServerAliveInterval=15 -o ServerAliveCountMax=3 "
              "-o ConnectTimeout=15 -o ExitOnForwardFailure=yes "
              f"-f -N -R {TUNNEL_PORT}:localhost:{LLM_PORT} colab@{VPS}")

def _tunnel_up():
    # End-to-end probe through the tunnel: if the forward is live, Colab 127.0.0.1:8898
    # reaches the local API. Immune to pgrep self-match on ANY platform (GNU or busybox).
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{TUNNEL_PORT}/healthz", timeout=3)
        return True
    except Exception:
        return False

def _open_tunnel():
    pat = "[%s]%s:localhost:%s" % (str(TUNNEL_PORT)[0], str(TUNNEL_PORT)[1:], LLM_PORT)
    subprocess.run("pkill -f '%s' 2>/dev/null" % pat, shell=True)
    time.sleep(1)
    r = subprocess.run(TUNNEL_CMD, shell=True, capture_output=True, text=True, timeout=30)
    return r.returncode == 0, r.stderr[:160]

def _tunnel_watchdog():
    while True:
        time.sleep(15)
        if not _tunnel_up():
            ok, err = _open_tunnel()
            print("[tunnel] re-established" if ok else "[tunnel] retry failed: " + err, flush=True)

if not globals().get("_TUNNEL_WATCH_STARTED"):
    _TUNNEL_WATCH_STARTED = True
    threading.Thread(target=_tunnel_watchdog, daemon=True).start()
    print("   tunnel watchdog armed (15s)", flush=True)

if not _tunnel_up():
    ok, err = _open_tunnel()
    if ok:
        time.sleep(2)   # let ssh -f fork + forward establish
        if _tunnel_up():
            print("   Tunnel up (verified)", flush=True)
        else:
            print("   Tunnel: ssh spawned but forward NOT established — watchdog will retry in 15s", flush=True)
    else:
        print("   Tunnel failed: " + err, flush=True)
else:
    print("   Tunnel already up", flush=True)

print("[3/4] Colony registration...")
def _colony_register():
    data = json.dumps({"id": MACHINE_ID, "gpu": gpu_name, "vram": gpu_vram,
        "ip": "colab-ephemeral", "capabilities": ["brain", "llm", "gpu", "shell"],
        "llm_endpoint": f"http://127.0.0.1:{TUNNEL_PORT}", "llm_model": MODEL_ALIAS}).encode()
    req = urllib.request.Request(f"{COLONY_URL}/colony/register", data=data,
        headers={"Content-Type": "application/json", "X-Auth-Token": TOKEN})
    return json.loads(urllib.request.urlopen(req, timeout=10).read())

try:
    print("   Registered:", json.dumps(_colony_register()))
except Exception as e:
    print("   Register failed:", e)

def _colony_keepalive():
    fails = 0
    while True:
        try:
            req = urllib.request.Request(f"{COLONY_URL}/colony/keepalive",
                data=json.dumps({"id": MACHINE_ID}).encode(),
                headers={"Content-Type": "application/json", "X-Auth-Token": TOKEN})
            urllib.request.urlopen(req, timeout=5)
            fails = 0
        except Exception:
            fails += 1
            if fails >= 3:
                try:
                    _colony_register()
                    print("[colony] re-registered after keepalive failures", flush=True)
                except Exception as e:
                    print("[colony] re-register failed:", e, flush=True)
                fails = 0
        time.sleep(60)
if not globals().get("_KEEPALIVE_STARTED"):
    _KEEPALIVE_STARTED = True
    threading.Thread(target=_colony_keepalive, daemon=True).start()
    print("   colony keepalive armed (60s)", flush=True)

In [ ]:
# @title 🧪 CELL 5: Gate self-test — READY FOR STRIX? (gate verified both ways)
import urllib.request, json, urllib.error

BASE = f"http://127.0.0.1:{LLM_PORT}"
_AUTH = {"Authorization": "Bearer " + (globals().get("BRAIN_API_KEY") or "")}
TUNNEL_PORT = globals().get("TUNNEL_PORT", 8898)   # defensive — normally set in Cell 1

def get(path, headers=None, timeout=5):
    req = urllib.request.Request(BASE + path, headers=headers or {})
    return urllib.request.urlopen(req, timeout=timeout)

def post(path, body):
    req = urllib.request.Request(BASE + path, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", **_AUTH})
    return urllib.request.urlopen(req, timeout=180)

def _status(fn):
    try:
        return fn()
    except Exception as e:
        return "FAIL " + repr(e)

def _expect(code, fn):
    """Pass if the call raises HTTPError with exactly `code` (urllib raises on non-2xx)."""
    try:
        return fn().status == code
    except urllib.error.HTTPError as e:
        return e.code == code
    except Exception:
        return False

results = {}
# 1. healthz OPEN (no auth) — liveness path must stay public
results["healthz"] = _status(lambda: get("/healthz").status)
# 2. GATE NEGATIVE: /v1/models WITHOUT auth must 401 (fail closed)
results["gate_noauth"] = _expect(401, lambda: get("/v1/models"))
# 3. GATE NEGATIVE: /v1/models with WRONG bearer must 401
results["gate_wrongkey"] = _expect(401, lambda: get("/v1/models", {"Authorization": "Bearer wrong-key"}))
# 4. GATE POSITIVE: /v1/models with real key lists the alias
results["models"] = _status(lambda: MODEL_ALIAS in [x["id"] for x in json.load(get("/v1/models", _AUTH))["data"]])
# 5. chat round-trip
results["chat"] = _status(lambda: (json.load(post("/v1/chat/completions", {"model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": "Reply with exactly: BRAIN_ALIVE"}],
    "temperature": 0.1, "max_tokens": 16}))["choices"][0]["message"].get("content") or "")[:40])
# 6. tool call round-trip
results["toolcall"] = _status(lambda: bool(json.load(post("/v1/chat/completions", {"model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": "What is the weather in Paris? Use the get_weather tool."}],
    "tools": [{"type": "function", "function": {"name": "get_weather", "description": "Weather for a city",
               "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}],
    "temperature": 0.1, "max_tokens": 64}))["choices"][0]["message"].get("tool_calls")))
# 7. SSE streaming round-trip
def _sse_test():
    r = post("/v1/chat/completions", {"model": MODEL_ALIAS,
        "messages": [{"role": "user", "content": "Count to 3."}], "temperature": 0.1, "max_tokens": 16, "stream": True})
    body = r.read().decode()
    return ("data: [DONE]" in body) and ("text/event-stream" in r.headers.get("content-type", ""))
results["sse"] = _status(_sse_test)
# 8. usage log wrote a real line with counts
def _usage_test():
    try:
        lines = open("/content/brain_usage.log").read().strip().splitlines()
    except Exception:
        return False
    return bool(lines) and "prompt=" in lines[-1] and "completion=" in lines[-1]
results["usage_log"] = _status(_usage_test)

print("\n" + "=" * 55)
print("  GATE SELF-TEST")
print("=" * 55)
for k, v in results.items():
    print(f"  {k:12s} {v}")
ok = (results.get("healthz") == 200
      and results.get("gate_noauth") is True
      and results.get("gate_wrongkey") is True
      and results.get("models") is True
      and isinstance(results.get("chat"), str) and results["chat"].startswith("BRAIN")
      and results.get("toolcall") is True
      and results.get("sse") is True
      and results.get("usage_log") is True)
print("=" * 55)
if ok:
    print("  READY FOR STRIX — all gates green (auth fail-closed verified)")
    print("  NOTE: production Strix pointer is DEEPSEEK (always-on). Brain = optional zero-quota 30B runs.")
    print("  Optional flip to brain on VPS:  bash /opt/chimera/brain_flip.sh on")
    print("  Back to DeepSeek:               bash /opt/chimera/brain_flip.sh off")
else:
    print("  NOT READY — fix the red items above, then re-run this cell")
